# AF2 spectral factorization — static and observability audit
Validation/train only. This notebook performs no training and never exposes test.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/kaggle/working'); REPO=WORK/'coffee-bean-detection'; INPUT=Path('/kaggle/input')
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
os.chdir(REPO)
sys.path.insert(0,str(REPO/'src'))
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'): sys.modules.pop(module_name,None)
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
DATA,ARTIFACTS,CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
print('DATA:',DATA); print('CONTRACT:',CONTRACT['decision'])

In [ ]:
from coffee_detector.af2_spectral.audit import run_spectral_static_audit
from coffee_detector.af2_spectral.observability import run_spectral_observability_audit
OUT=WORK/'af2-spectral-static-audit'; OUT.mkdir(exist_ok=True)
cpu_static=run_spectral_static_audit(ARTIFACTS['D0_seed42_best.pt'],OUT/'static_audit_cpu.json',device='cpu')
static=run_spectral_static_audit(ARTIFACTS['D0_seed42_best.pt'],OUT/'static_audit_gpu.json',device='cuda:0')
assert cpu_static['decision']==static['decision']=='PASS', 'STOP: static audit CPU/GPU gagal; training dilarang.'
observability=run_spectral_observability_audit(DATA,OUT/'observability_train.json',seed=42,device='cuda:0')
assert observability['test_images_accessed'] is False
print('STATIC CPU/GPU:',static['decision']); print('OBSERVABILITY:',OUT/'observability_train.json')

In [ ]:
archive=shutil.make_archive('/kaggle/working/af2-spectral-static-audit','zip',OUT)
print('DOWNLOAD SEBELUM STOP SESSION:',archive)